<a href="https://colab.research.google.com/github/Layhok14/GEOAI_Cambodia_Potential_Dataset/blob/main/guides/linked/ee-api-colab-setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install geemap earthengine-api osmnx geopandas folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.8 MB/s eta 0:00:00


### Import the API

Run the following cell to import the API into your session.

In [2]:
import ee

In [25]:
import geemap
import osmnx as ox
import folium

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [5]:
# Trigger the authentication flow.
ee.Authenticate()

In [8]:
# Initialize the library.
ee.Initialize(project='project-c58dc061-9219-48f1-acb')

### DEFINE CAMBODIA REGION OF INTEREST

In [26]:
cambodia_poi = ee.Geometry.Point([104.9282, 11.5564]) # Phnom Penh
cambodia_bounds = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Cambodia'))

In [27]:
Map = geemap.Map(center=[12.5657, 104.9910], zoom=7)
Map.add_basemap('SATELLITE')

In [16]:
print(Map)

Map(center=[12.5657, 104.991], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transparent_bg=True, widget=GridBox(children=(<geemap.map_widgets.LayerManager object at 0x7e462825ee90>, <geemap.toolbar.Toolbar object at 0x7e462825efd0>), layout=Layout(grid_gap='0px 10px', grid_template_columns='auto auto', overflow='visible'))), WidgetControl(options=['position', 'transparent_bg'], transparent_bg=True, widget=<geemap.map_widgets.SearchBar object at 0x7e462825e5d0>), ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text', 'zoom_out_title']), FullScreenControl(options=['position']), MapDrawControl(marker={'shapeOptions': {'color': '#3388ff'}}, options=['position'], polygon={'shapeOptions': {}}, polyline={'shapeOptions': {}}, rectangle={'shapeOptions': {'color': '#3388ff'}}), ScaleControl(options=['imperial', 'max_width', 'metric', 'position', 'update_when_idle'], position='bottomleft'), MeasureControl(active_color='orange', o

### LOAD & VISUALIZE GEE DATASETS

In [18]:
# 1. Sentinel-2
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(cambodia_bounds) \
    .filterDate('2023-01-01', '2023-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .median().clip(cambodia_bounds)
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 (RGB)', False)

In [21]:
# 2. VIIRS
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
    .filterDate('2023-01-01', '2023-12-31') \
    .select('avg_rad').median().clip(cambodia_bounds)
Map.addLayer(viirs, {'min': 0, 'max': 10, 'palette': ['black', 'blue', 'purple', 'yellow', 'white']}, 'VIIRS Night Lights')

In [22]:
# 3. WorldPop (Population Density)
worldpop = ee.ImageCollection("WorldPop/GP/100m/pop") \
    .filterBounds(cambodia_bounds).filterDate('2020-01-01', '2021-01-01') \
    .mosaic().clip(cambodia_bounds)
Map.addLayer(worldpop, {'min': 0, 'max': 50, 'palette': ['#24126c', '#1fff4f', '#d4ff50']}, 'WorldPop Density', False)

In [28]:
# 4. ESA WorldCover (10m Land Cover)
esa_lc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(cambodia_bounds)
Map.addLayer(esa_lc, {'bands': ['Map']}, 'ESA WorldCover', False)

In [29]:
# 5. SRTM DEM (Elevation)
srtm = ee.Image('USGS/SRTMGL1_003').clip(cambodia_bounds)
Map.addLayer(srtm, {'min': 0, 'max': 1500, 'palette': ['#006600', '#E5E500', '#E59900', '#B22222', '#FFFFFF']}, 'SRTM Elevation', False)

In [31]:
# 6. CHIRPS (Rainfall/Precipitation - Annual Sum)
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
    .filterDate('2023-01-01', '2023-12-31') \
    .sum().clip(cambodia_bounds)
Map.addLayer(chirps, {'min': 1000, 'max': 3000, 'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494']}, 'CHIRPS Annual Rainfall', False)

In [32]:
# 7. Landsat 9 (30m)
l9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
    .filterBounds(cambodia_bounds) \
    .filterDate('2023-01-01', '2023-12-31') \
    .median().clip(cambodia_bounds)
# Apply scaling factors for Landsat 9 surface reflectance
def apply_scale_factors(image):
    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)
l9_scaled = apply_scale_factors(l9)
Map.addLayer(l9_scaled, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}, 'Landsat 9 (RGB)', False)

### Third-party library

In [33]:
# Add Mapillary Street-Level Imagery Coverage Tile Layer
mapillary_tiles = "https://raster-tiles.mapillary.com/v0.1/{z}/{x}/{y}.png"
Map.add_tile_layer(mapillary_tiles, name="Mapillary Street-Level Coverage", attribution="Mapillary", shown=False)

In [34]:
# Add Ookla Global Internet Speed Tiles (Mobile Download)
ookla_tiles = "https://c.tile.openstreetmap.org/{z}/{x}/{y}.png" # Placeholder map, replace with Ookla quadkey tiles in advanced workflows
# Map.add_tile_layer(...)

In [37]:
# OSMnx VECTOR DATA (Roads/Buildings)
# Due to memory limits, we only fetch for a small bounding box in Phnom Penh.
print("Fetching OSM Data for central Phnom Penh (this takes a moment)...")
try:
    pp_center = (11.5564, 104.9282) # Lat, Lon
    # Fetch road network graph
    G = ox.graph_from_point(pp_center, dist=2000, network_type='drive')
    # Convert graph to GeoDataFrames
    nodes, edges = ox.graph_to_gdfs(G)

    # Add OSM roads to the map as a vector overlay
    Map.add_gdf(edges, layer_name="OSM Roads (Phnom Penh)", style={'color': 'red', 'weight': 1})
    print("OSM Data successfully loaded!")
except Exception as e:
    print(f"OSM Fetch failed: {e}")

Fetching OSM Data for central Phnom Penh (this takes a moment)...
OSM Data successfully loaded!


### Render the map

In [39]:
Map.addLayerControl()
Map

Map(bottom=22633.0, center=[49.621387109259516, 335.6735229492188], controls=(WidgetControl(options=['position…